# MLflow: 5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/mlops'
import os, shutil
assert os.path.isdir(DRIVE), f"مش لاقي {DRIVE}"
WORK = '/content/project'
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK + '/data')
for name in ['src', 'configs', 'scripts']:
    shutil.copytree(f'{DRIVE}/{name}', f'{WORK}/{name}')
shutil.copy(f'{DRIVE}/requirements.txt', WORK)
shutil.copytree(f'{DRIVE}/processed', f'{WORK}/data/processed')
os.chdir(WORK)
if os.path.exists(f'{DRIVE}/mlflow.db'):
    shutil.copy(f'{DRIVE}/mlflow.db', f'{WORK}/mlflow.db')
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
print("تمام")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
تمام


In [ ]:
!pip install -q 'transformers>=4.46' accelerate scikit-learn pandas pyyaml mlflow

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "مفيش GPU! غيّر الـRuntime لـT4 وشغّل من الأول"

GPU: True Tesla T4


## Experiments


In [ ]:
SAMPLES = 60000
RUNS = [1, 2, 3, 4, 5]
CONFIGS = {1: (2e-5, 8, 2), 2: (3e-5, 8, 2), 3: (5e-5, 8, 3), 4: (3e-5, 16, 2), 5: (3e-5, 16, 3)}  # lr, batch, epochs
TRACKING = f"sqlite:///{WORK}/mlflow.db"

import time
for i in RUNS:
    lr, bs, ep = CONFIGS[i]
    drive_run = f'{DRIVE}/runs/run_{i}'
    if os.path.exists(f'{drive_run}/metrics.json'):
        print(f"run {i}: خلصت قبل كده، هعدّيها")
        continue
    out = f'models/run_{i}'
    name = f'run{i}_lr{lr}_bs{bs}_ep{ep}'
    print(f"\n===== {name} =====")
    t = time.time()
    cmd = (f"python -m src.train --max-samples {SAMPLES} --learning-rate {lr} --batch-size {bs} --epochs {ep} "
           f"--output-dir {out} --model-version run{i} --run-name {name} --experiment arabic-sentiment --tracking-uri {TRACKING}")
    !{cmd}
    assert os.path.exists(f'{out}/metrics.json'), f"run {i} فشلت، شوف الـerror اللي فوق"
    shutil.copytree(out, drive_run, ignore=shutil.ignore_patterns('checkpoints'))
    shutil.copy(f'{WORK}/mlflow.db', f'{DRIVE}/mlflow.db')
    print(f"run {i} خلصت في {(time.time()-t)/60:.1f} دقيقة وانحفظت في Drive")

Streaming output truncated to the last 5000 lines.
    "max_length": 128,
    "seed": 42,
    "max_samples": 60000
  }
}
MLflow run: 630177cc2c3e4da79b10ca8a88e6c5d5
run 2 خلصت في 23.8 دقيقة وانحفظت في Drive

===== run3_lr5e-05_bs8_ep3 =====
Loading weights: 100% 199/199 [00:00<00:00, 1418.41it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weigh

## Result and the Best Choice

In [ ]:
import mlflow, json
mlflow.set_tracking_uri(TRACKING)
df = mlflow.search_runs(experiment_names=['arabic-sentiment'])
cols = {'tags.mlflow.runName': 'run', 'params.learning_rate': 'lr', 'params.batch_size': 'bs', 'params.epochs': 'epochs',
        'metrics.val_f1_macro': 'val_f1', 'metrics.test_f1_macro': 'test_f1', 'metrics.test_accuracy': 'test_acc',
        'metrics.f1_neutral': 'f1_neutral', 'metrics.train_seconds': 'seconds'}
table = df[list(cols)].rename(columns=cols).sort_values('val_f1', ascending=False).reset_index(drop=True)
display(table.round(4))

best = table.iloc[0]['run']
idx = int(best.split('_')[0].replace('run', ''))
dst = f'{DRIVE}/models/v2'
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(f'{DRIVE}/runs/run_{idx}', dst)

with open(f'{dst}/metrics.json', 'r', encoding='utf-8') as f:
    m = json.load(f)

m['model_version'] = 'v2'

with open(f'{dst}/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(m, f, indent=2, ensure_ascii=False)

shutil.copy(f'{WORK}/mlflow.db', f'{DRIVE}/mlflow.db')
print(f"\nالأفضل: {best}  ->  اتحفظ في Drive/mlops/models/v2")

,run,lr,bs,epochs,val_f1,test_f1,test_acc,f1_neutral,seconds
0,run4_lr3e-05_bs16_ep2,3e-05,16,2,0.8529,0.8474,0.8979,0.7580,923.1
1,run2_lr3e-05_bs8_ep2,3e-05,8,2,0.8513,0.8470,0.8971,0.7568,1250.6
2,run1_lr2e-05_bs8_ep2,2e-05,8,2,0.8494,0.8468,0.8975,0.7562,1251.9
3,run5_lr3e-05_bs16_ep3,3e-05,16,3,0.8489,0.8460,0.8960,0.7577,1381.7
4,run3_lr5e-05_bs8_ep3,5e-05,8,3,0.8423,0.8381,0.8918,0.7449,1919.2



الأفضل: run4_lr3e-05_bs16_ep2  ->  اتحفظ في Drive/mlops/models/v2
